# PPE dataset exploration

Run this notebook after `backend/scripts/download_dataset.py` has
populated `backend/data/`. It visualizes class balance and a sample
grid of annotated images to sanity-check the dataset before training.

In [ ]:
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import yaml

BACKEND_DIR = Path.cwd().parent / "backend"
DATA_YAML = BACKEND_DIR / "data" / "data.yaml"

with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

class_names = data_cfg["names"]
dataset_root = Path(data_cfg["path"])
print("Classes:", class_names)

In [ ]:
# Count instances per class across the train split label files.
labels_dir = dataset_root / "train" / "labels"
counter = Counter()

for label_file in labels_dir.glob("*.txt"):
    for line in label_file.read_text().splitlines():
        if not line.strip():
            continue
        class_id = int(line.split()[0])
        counter[class_names[class_id]] += 1

names, counts = zip(*sorted(counter.items(), key=lambda x: -x[1])) if counter else ([], [])
plt.figure(figsize=(9, 4))
plt.bar(names, counts)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Instance count")
plt.title("Class distribution (train split)")
plt.tight_layout()
plt.show()

In [ ]:
# Show a grid of sample images with their raw YOLO-format bounding boxes.
import random

import cv2

images_dir = dataset_root / "train" / "images"
sample_images = random.sample(list(images_dir.glob("*.jpg")), k=min(6, len(list(images_dir.glob("*.jpg")))))

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, image_path in zip(axes.flat, sample_images):
    image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]
    label_path = labels_dir / f"{image_path.stem}.txt"
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            class_id, cx, cy, bw, bh = (float(v) for v in line.split())
            x1 = int((cx - bw / 2) * w)
            y1 = int((cy - bh / 2) * h)
            x2 = int((cx + bw / 2) * w)
            y2 = int((cy + bh / 2) * h)
            cv2.rectangle(image, (x1, y1), (x2, y2), (255, 0, 0), 2)
    ax.imshow(image)
    ax.axis("off")
plt.tight_layout()
plt.show()